# MB-Fit tutorial (v20190924)

This notebook will walk you through the multiple possibilities one has to obtain many-body fits for multiple molecules. 



## Chapter 0. Set up the notebook.

### 0.1. Import the python library
Remember that in order to import the library without any errors, you need to perform the following operations in the bash terminal from which you are running the notebook. If you didn't do it, please, close the notebook and write in a bash terminal:
```sh
cd HOME/DIRECTORY/OF/POTENTIAL_FITTING
source install.sh
```
Now the following command should run without any problem.

In [1]:
# This is for testing purposes. Can be ignored.
%load_ext autoreload
%autoreload 2

In [2]:
# The library that will enable the fitting generation and energy calculation
import potential_fitting
# Some other useful libraries
import os

## Example 3. Generate a CO2-CO2 two-body MB-nrg PEF

### 3.1. Definition of the variables

In [3]:
main_dir = os.getcwd()

In [4]:
# The software that will be used to perform all the calculations
code = "qchem"
#code = "psi4"

# The quantum chemistry method we want to use
method = "HF"
#method = "MP2"
#method = "wb97m-v"

# Basis set to use. Must be pre-defined in the software. Custom basis sets not implemented yet.
basis = "STO-3G"

# Use counter-poise correction or not.
cp = False
#cp = True

# Number of threads and memory we would like to use
num_threads = 2
memory = "4GB"

# This is the path where all the log files will be stored.
log_path = "logs"

In [5]:
# Names that will identify the monomers. This is used for identification purposes only.
names = ["P","P","P"]

# Number of atoms of each monomer
number_of_atoms = [1,1,1]

# Charge of each monomer
charges = [0,0,0]

# Spin multiplicity of each monomer
spin = [1,1,1]

# Use MB-pol for water (if applicable). 
# If 1 will use the Partridge-Shwenke PEF for water, with the position dependent charges.
use_mbpol = [0,0,0]

In [6]:
# Symmetry of the molecule
symmetry = ["A1", "A1", "A1"]

# SMILES string
smiles = ["P", "P", "P"]

In [7]:
# Settings for monomer
mon_settings = "monomer_settings.ini"

my_settings_file_mon = """
[files]
# Local path directory to write log files in
log_path = """ + log_path + """

[config_generator]
# what library to use for geometry optimization and normal mode generation
code = """ + code + """
# use geometric or linear progression for T and A in config generation, exactly 1 must be True
geometric = False
linear = False

[energy_calculator]
# what library to use for energy calculations
code = """ + code + """

[psi4]
# memory to use when doing a psi4 calculation
memory = """ + memory + """
# number of threads to use when executing a psi4 calculation
num_threads = """ + str(num_threads) + """

[qchem]
# number of threads to use when executing a qchem calculation
num_threads = """ + str(num_threads) + """

[molecule]
# name of fragments, seperated by commas
names = """ + names[0] + """
# number of atoms in each fragment, seperated by commas
fragments = """ + str(number_of_atoms[0]) + """
# charge of each fragment, seperated by commas
charges = """ + str(charges[0]) + """
# spin multiplicity of each fragment, seperated by commas
spins = """ + str(spin[0]) + """
# tag when putting geometries into database
tag = none
# Use or not MB-pol
use_mbpol = """ + str(use_mbpol[0]) + """
# symmetry of each fragment, seperated by commas
symmetry = """ + symmetry[0] + """
SMILES = """ + smiles[0] + """
"""



In [8]:
# Settings for trimer
trim_settings = "trimer_settings.ini"

my_settings_file_trim = """
[files]
# Local path directory to write log files in
log_path = """ + log_path + """

[config_generator]
# what library to use for geometry optimization and normal mode generation
code = """ + code + """
# use geometric or linear progression for T and A in config generation, exactly 1 must be True
geometric = False
linear = False

[energy_calculator]
# what library to use for energy calculations
code = """ + code + """

[psi4]
# memory to use when doing a psi4 calculation
memory = """ + memory + """
# number of threads to use when executing a psi4 calculation
num_threads = """ + str(num_threads) + """

[qchem]
# number of threads to use when executing a qchem calculation
num_threads = """ + str(num_threads) + """

[molecule]
# name of fragments, seperated by commas
names = """ + names[0] + "," + names[1] + "," + names[2] + """
# number of atoms in each fragment, seperated by commas
fragments = """ + str(number_of_atoms[0]) + """,""" + str(number_of_atoms[1]) + """,""" + str(number_of_atoms[2]) + """
# charge of each fragment, seperated by commas
charges = """ + str(charges[0]) + """,""" + str(charges[1]) + """,""" + str(charges[2]) + """
# spin multiplicity of each fragment, seperated by commas
spins = """ + str(spin[0]) + """,""" + str(spin[1]) + """,""" + str(spin[2]) + """
# tag when putting geometries into database
tag = none
# Use or not MB-pol
use_mbpol = """ + str(use_mbpol[0]) + """,""" + str(use_mbpol[1]) + """,""" + str(use_mbpol[2]) + """
# symmetry of each fragment, seperated by commas
symmetry = """ + symmetry[0] + """,""" + symmetry[1] + """,""" + symmetry[2] + """
SMILES = """ + smiles[0] + """,""" + smiles[1] + """,""" + smiles[2] + """
"""

In [9]:
# Write the files:
ff = open(mon_settings,'w')
ff.write(my_settings_file_mon)
ff.close()

ff = open(trim_settings,'w')
ff.write(my_settings_file_trim)
ff.close()

In [10]:
# XYZ file that contains the unoptimized geommetry of the monomer
unopt_mon = "monomer.xyz"

my_unopt_monomer = """1
unoptimized P
P   0   0   0
"""

In [11]:
# Write the file:
ff = open(unopt_mon,'w')
ff.write(my_unopt_monomer)
ff.close()

In [12]:
# XYZ file that contains the optimized geommetry of the monomer
opt_mon = unopt_mon

# Same for dimer
unopt_trim = "trimer.xyz"
opt_trim = unopt_trim

In [13]:
# XYZ file with the configurations of the training set
rigid_training_configs = "rigid_training_configs.xyz" 
flex_training_configs = "flex_training_configs.xyz"
normal_mode_training_configs = "normal_mode_training_configs"

ttm_training_configs = "ttm_training_configs.xyz"

# XYZ file with the configurations of the test set
rigid_test_configs = "rigid_test_configs.xyz" 
flex_test_configs = "flex_test_configs.xyz"
normal_mode_test_configs = "normal_mode_test_configs"

ttm_test_configs = "ttm_test_configs.xyz"

# Distorted monomer configurations for the flexible training set
mon_distorted = "mon_distorted.xyz"

# And the screened values
mon_screened = "mon_screened.xyz"

# XYZ file with the training set that the codes need to perform the fit
# Configurations are the same as training_configs but this file
# has the energies in the comment line
training_set = "training_set.xyz"
ttm_training_set = "ttm_training_set.xyz"

# XYZ file with the test set that the codes need to perform the fit
# Configurations are the same as test_configs but this file
# has the energies in the comment line 
test_set = "test_set.xyz"
ttm_test_set = "ttm_test_set.xyz"


In [14]:
# Monomers 1 and 2 separated by '_'
molecule_in = "_".join(symmetry)

# Configuration file that contains all the monomer 
# and dimer information. Will be used to generate the 2B codes.
config = "config.ini"

# Input file for the polynomial generation
poly_in = "poly.in"

# Directory where the polynomials will be generated
poly_directory = "polynomial_generation"

# Degree of the polynomials
polynomial_order = 4

In [15]:
# Directory where mb-nrg fitting code will be stored
mbnrg_directory = "mb-nrg_fitting_code"
mbnrg_fits_directory = "mb-nrg_fits"

In [16]:
# Number of configurations in the 2b training_set
num_training_configs = 10000              
############################

# Number of configurations in the 2b test set
num_test_configs = int(0.2*num_training_configs)

############################

# Number of distorted configurations for monomer 1 and monomer 2
num_mon_distorted = 100

# Maximum energy allowed for distorted monomers (in kcal/mol)
mon_emax = 30.0

# Maximum binding energy allowed
bind_emax = 500.0

# Minimum and maximum distance between the two monomers !cutoff
min_d_2b = 6.0
max_d_2b = 14.0

# Minimum fraction of the VdW distance that is allowed between any atoms that belong to different monomers
min_inter_d = 0.5

# Seeds to be used in the configuration generation to ensure different
# configurations for training and test
seed_training = 23410
seed_test = 93109

# IDs of the monomers (should be consistent with the 1B id for each)
mon_ids = ["p","p","p"]

# Number of MB-nrg fits to perform
num_mb_fits = 40

### 3.2. Generate polynomials

#### 3.2.1. Generate polynomial input file

In [17]:
help(potential_fitting.generate_poly_input)

Help on function generate_poly_input in module potential_fitting.potential_fitting:

generate_poly_input(settings_path, molecule_in, in_file_path, virtual_sites=['X', 'Y', 'Z'])
    Generates an input file for polynomial generation.
    
    Args:
        settings_path       - Local path to the file containing all relevent settings information.
        molecule_in         - String idicating symmetry of molecule, ie "A1B2_A1B2" for (CO2)2
        in_file_path        - Local path to the file to write the polynomial input to.
        virtual_sites       - List of Symmetry labels that are virtual sites.
                Default: ["X", "Y", "Z"]
    
    Returns:
        None.



In [18]:
potential_fitting.generate_poly_input(trim_settings, molecule_in, poly_in)

Generating polynomial input file for symmetry A1_A1_A1 into file poly.in.
Adding filter to filter out terms that ONLY use intramolecular variables.
Successfully generated polynomial input file! 3 total variables.


#### 3.2.2. Generate polynomial files

In [19]:
help(potential_fitting.generate_polynomials)

Help on function generate_polynomials in module potential_fitting.potential_fitting:

generate_polynomials(settings_path, poly_in_path, order, poly_dir_path, generate_direct_gradients=True, num_gradient_terms_per_line=1)
    Generates polynomial input for maple and some ".cpp" and ".h" polynomial files.
    
    Args:
        settings_path       - Local path to the file containing all relevent settings information.
        poly_in_path        - Local path to the file to read the polynomial input from. Name of file should be in
                the format "A1B2.in". It is ok to have extra directories prior to file name. For example:
                "thisplace/thatplace/A3.in".
        order               - The order of the polynomial to generate.
        poly_dir_path       - Local path to the directory to write the polynomial files in.
        generate_direct_gradients - If True, then a gradients cpp file is generate additionally to the polynomial
                cpp file. This may take

In [20]:
potential_fitting.generate_polynomials(trim_settings, poly_in, polynomial_order, poly_directory, generate_direct_gradients=True)

Generating polynomial files from input file poly.in into directory polynomial_generation.
Parsing polynomial input file poly.in...
Atom names ['A1a', 'A2b', 'A3c']
File polynomial_generation/poly.log already exists, moving existing polynomial_generation/poly.log to polynomial_generation/poly.log.backup-32 to make way for new file.
Finding permutations...
Generating terms up to degree 4...
|====================================================================================================|
1 possible degree 1 terms, now filtering them...
There were 1 accepted degree 1 terms.
2 possible degree 2 terms, now filtering them...
There were 2 accepted degree 2 terms.
3 possible degree 3 terms, now filtering them...
There were 3 accepted degree 3 terms.
4 possible degree 4 terms, now filtering them...
There were 4 accepted degree 4 terms.
There were 10 accepted terms over all
Writing .h and .maple polynomial files in directory polynomial_generation...
Writing gradients...
|====================

#### 3.2.3. Optimize the polynomial evaluation

In [21]:
help(potential_fitting.execute_maple)

Help on function execute_maple in module potential_fitting.potential_fitting:

execute_maple(settings_path, poly_dir_path)
    Runs maple on the polynomial files in the specified directory to turn them into actual cpp files.
    
    Args:
        settings_path       - Local path to the file containing all relevent settings information.
        poly_directory      - Local path to the directory to read the ".maple" files from and write the ".cpp" files
                to.
    
    Returns:
        None.



In [22]:
#potential_fitting.execute_maple(dim_settings, poly_directory)

### 3.3. Geometry optimization and normal mode calculation

#### 3.3.1. Monomers

In [23]:
#help(potential_fitting.optimize_geometry)

In [24]:
# Optimize monomer
#potential_fitting.optimize_geometry(mon_settings, unopt_mon, opt_mon, method, basis)

In [25]:
#help(potential_fitting.generate_normal_modes)

In [26]:
# Get its normal modes
#potential_fitting.generate_normal_modes(mon_settings, opt_mon,normal_modes_mon, method, basis)

#### 3.3.2. Dimer

Now the same for the dimer.

In [27]:
#help(potential_fitting.generate_2b_configurations)

In [28]:
# Generate a dimer
#potential_fitting.generate_2b_configurations(dim_settings, opt_mon, opt_mon, 
#                                             1, unopt_dim, 
#                                             min_distance = 2, max_distance = 5, 
#                                             min_inter_distance = 0.8, 
#                                             progression=False, use_grid=False, 
#                                             step_size=0.5, num_attempts=100, 
#                                             logarithmic=True, distribution=None, 
 #                                            mol1_atom_index=None, mol2_atom_index=None, 
#                                             seed=seed_training)

In [29]:
# Optimize the dimer
#potential_fitting.optimize_geometry(dim_settings, unopt_dim, opt_dim, method, basis)

In [30]:
# Get its normal modes
#potential_fitting.generate_normal_modes(dim_settings, opt_dim,normal_modes_dim, method, basis)

### 3.5. Obtain config file

In [31]:
# C6, A and b parameters are obtained from example 2
#example2_c6 = [319.9415, 221.5987, 173.3298]
#example2_d6 = [3.08949, 3.71685, 4.09252]
#example2_a = [15312.3, 20732.5, 78777.2]

In [32]:
#help(potential_fitting.get_system_properties)

In [33]:
#chg, pol, c6 = potential_fitting.get_system_properties(dim_settings, config, geo_paths = [opt_mon,opt_mon])
chg = [[0.0],[0.0],[0.0]]
pol = [[0.0],[0.0],[0.0]]
c6 = [0.0]
a = [0.0]
d6 = [0.0]

In [34]:
help(potential_fitting.write_config_file)

Help on function write_config_file in module potential_fitting.potential_fitting:

write_config_file(settings_file, config_path, charges, pols, geo_paths, C6=[0.0], polfacs=None, d6=None, A=None, kmin=0.0, kmax=50.0, dmin=0.0, dmax=50.0, bmin=0.0, bmax=10.0, kmin_init=1.0, kmax_init=4.0, dmin_init=1.0, dmax_init=4.0, bmin_init=1.0, bmax_init=4.0, r_in=7.0, r_out=8.0, energy_range=20, alpha=0.0005, virtual_sites_label=['X', 'Y', 'Z'], var_intra='exp', var_inter='exp', var_virtual_sites='coul')
    Writes the config file.
    
    Qchem is required for this step to work.
    
    Args:
        settings_path       - Local path to the file containing all relevent settings information.
        config_path         - Local path to file to write the config file to.
        charges             - List with the charges of each monomer [[mon1],[mon2],..]
        pols                - List with the pols of each monomer [[mon1],[mon2],..]
        geo_paths           - List of local paths to the opti

In [35]:
os.chdir(main_dir)
potential_fitting.write_config_file(trim_settings, config, chg, pol, 
                                    [opt_mon, opt_mon, opt_mon], C6 = c6, 
                                    d6=d6, A=a, var_inter= 'exp0', r_in=10, r_out=12)
# set rin/rout cutoff for trimer

File ./config.ini already exists, moving existing ./config.ini to ./config.ini.backup-68 to make way for new file.
Completed generating config file ./config.ini.


### 3.7. MB-nrg Training and Test Set generation

#### 3.7.1. Rigid Training Set

##### Generate configurations

In [36]:
# Training Set
#potential_fitting.generate_2b_configurations(dim_settings, opt_mon, opt_mon, 
#                                             num_training_configs, rigid_training_configs, 
#                                             min_distance = min_d_2b, max_distance = max_d_2b, 
#                                             min_inter_distance = min_inter_d, 
#                                             progression=True, use_grid=False, 
#                                             step_size=0.5, num_attempts=100, 
#                                             logarithmic=True, distribution=None, 
#                                             mol1_atom_index=None, mol2_atom_index=None, 
#                                             seed=seed_training)

In [37]:
# Test Set
#potential_fitting.generate_2b_configurations(dim_settings, opt_mon, opt_mon, 
#                                             num_rigid_test_configs, rigid_test_configs, 
#                                             min_distance = min_d_2b, max_distance = max_d_2b, 
#                                             min_inter_distance = min_inter_d, 
#                                             progression=True, use_grid=False, 
#                                             step_size=0.5, num_attempts=100, 
#                                             logarithmic=True, distribution=None, 
#                                             mol1_atom_index=None, mol2_atom_index=None, 
#                                             seed=seed_training)

##### Add configurations to the database

In [38]:
#help(potential_fitting.init_database)

In [39]:
# Training set
#potential_fitting.init_database(dim_settings, database_config, rigid_training_configs, 
#                                method, basis, cp, "train_rig_ex3", optimized = False)

In [40]:
# Test Set
#potential_fitting.init_database(dim_settings, database_config, rigid_test_configs, 
#                                method, basis, cp, "test_rig_ex3", optimized = False)

In [41]:
# Add monomer optimized geommetry to database (needed for binding energy)
#potential_fitting.init_database(mon_settings, database_config, opt_mon, method, basis, cp, "train_rig_ex3", optimized = True)
#potential_fitting.init_database(mon_settings, database_config, opt_mon, method, basis, cp, "test_rig_ex3", optimized = True)

#### 3.7.2. Flexible Configurations

##### Generate distorted monomer configurations

### 3.8. MB-nrg fit

#### 3.8.1. Obtain and compile the fitting code

In [42]:
help(potential_fitting.generate_mbnrg_fitting_code)

Help on function generate_mbnrg_fitting_code in module potential_fitting.potential_fitting:

generate_mbnrg_fitting_code(settings_path, config_path, poly_in_path, poly_path, poly_order, fit_dir_path, use_direct=False)
    Generates the fit code based on the polynomials for a system
    
    Args:
        settings_path       - Local path to the file containing all relevent settings information.
        config_path         - Local path to the dimer config file.
        poly_in_path        - Local path to the the A3B2.in type file to read polynomial input from.
        poly_path           - Local path to directory where polynomial files are.
        poly_order          - The order of the polynomial in poly_path.
        fit_dir_path        - Local path to directory to generate fit code in.
        use_direct          - If true, it will use the direct polynomials instead of the mapleoptimized
    
    Returns:
        None.



In [43]:
os.chdir(main_dir)
potential_fitting.generate_mbnrg_fitting_code(trim_settings, config, 
                                              poly_in, poly_directory, 
                                              polynomial_order, mbnrg_directory, 
                                              use_direct=True)

File ./config.ini already exists, moving existing ./config.ini to ./config.ini.backup-69 to make way for new file.
Executing python generator script
Generating fitcode for molecule with symmetry A1_A1_A1...
Using mbpol for 0 fragments.
0 fragments have lone pairs.
Using exp variables for intramolecular interactions.
Using exp0 variables for intermolecular interactions.
Using coul variables for lone pair interactions.
10 terms in the polynomial.
Atoms in the molecule: A1 A2 A3.
3 variables in the polynomial.
2 non-linear parameters in the polynomial.


In [44]:
help(potential_fitting.compile_fit_code)

Help on function compile_fit_code in module potential_fitting.potential_fitting:

compile_fit_code(settings_path, fit_dir_path)
    Compiles the fit code in the given directory.
    
    Args:
        settings_path       - Local path to the file containing all relevent settings information.
        fit_dir_path        - Local path to the directory to read uncompiled fit code from and write compiled fit code
                to.
    
    Returns:
        None.



In [45]:
os.chdir(main_dir)
potential_fitting.compile_fit_code(trim_settings, mbnrg_directory)

Compiling fit code...
Fit code compilation successful!


#### 3.8.2. Perform the fit

In [46]:
help(potential_fitting.prepare_fits)

Help on function prepare_fits in module potential_fitting.potential_fitting:

prepare_fits(settings_path, fit_dir_path, training_set_path, fits_path, DE=20, alpha=0.0005, num_fits=10, ttm=False, over_ttm=False)
    Prepares fits to be run by creating directories with executable scripts in them.
    
    Each directory will contain a run_fit.sh script, which when ran will run one fit.
    
    The directories will appear in the directory with your other log files.
    
    Args:
        settings_path       - Local path to the file containing all relevent settings information.
        fit_dir_path        - Local path to the directory containing the compiled fitcode.
        training_set_path   - Local path to training set to use for all the fits.
        fits_path           - Local path to the directory to create the fits in.
        DE                  - Low DE places more weight on low energy training set items. 
                              Large DE places even weight on all training

In [47]:
os.chdir(main_dir)
os.system('rm -r '+ mbnrg_fits_directory)
potential_fitting.prepare_fits(trim_settings, mbnrg_directory, 
                               training_set, mbnrg_fits_directory, 
                               DE=10, alpha=0.0005, num_fits=num_mb_fits, 
                               ttm=False, over_ttm=False)
# W =(DE/(E-Emin+DE))^2  where  DE ~ 2-5 times Emin
# higher value for DE (try to adjust)
# low alpha any value / ridge regresion, tikonov regularition

Succesfully created fit folder fit1.
Succesfully created fit folder fit2.
Succesfully created fit folder fit3.
Succesfully created fit folder fit4.
Succesfully created fit folder fit5.
Succesfully created fit folder fit6.
Succesfully created fit folder fit7.
Succesfully created fit folder fit8.
Succesfully created fit folder fit9.
Succesfully created fit folder fit10.
Succesfully created fit folder fit11.
Succesfully created fit folder fit12.
Succesfully created fit folder fit13.
Succesfully created fit folder fit14.
Succesfully created fit folder fit15.
Succesfully created fit folder fit16.
Succesfully created fit folder fit17.
Succesfully created fit folder fit18.
Succesfully created fit folder fit19.
Succesfully created fit folder fit20.
Succesfully created fit folder fit21.
Succesfully created fit folder fit22.
Succesfully created fit folder fit23.
Succesfully created fit folder fit24.
Succesfully created fit folder fit25.
Succesfully created fit folder fit26.
Succesfully created f

In [48]:
help(potential_fitting.execute_fits)

Help on function execute_fits in module potential_fitting.potential_fitting:

execute_fits(settings_path, fits_path)
    Looks for fit executables in fit directories in the logs directory and runs the fits.
    
    Will ignore any fits that already have a log file - suggesting they have already been run.
    
    Args:
        settings_path       - Local path to the file containing all relevent settings information.
        fits_path           - Local path to the directory to create the fits in.
    
    Returns:
        None.



In [49]:
potential_fitting.execute_fits(trim_settings, mbnrg_fits_directory)

fit36 is running.



CommandExecutionError: [1m[3m[31mThe following error occured in the Potential Fitting Library: Error when executing system command './run_fit.sh': entire command: './run_fit.sh', error: b'./run_fit.sh: line 3: 15907 Aborted                 (core dumped) /oasis/projects/nsf/csd497/yz393/Multi_body_fitting/MB-Fit-master/examples/NP_Dimer/mb-nrg_fitting_code/bin/fit-3b training_set.xyz 10 0.0005 > fit.log 2> fit.err\n' (exit code 134)[0m[0m[0m

In [ ]:
help(potential_fitting.retrieve_best_fit)

In [ ]:
potential_fitting.retrieve_best_fit(trim_settings, mbnrg_fits_directory, fitted_nc_path = "mbnrg.nc")

### 3.9 Visuzalize the fit

In [ ]:
help(potential_fitting.get_correlation_data)

In [ ]:
energies = potential_fitting.get_correlation_data(trim_settings, mbnrg_directory, 
                                                  mbnrg_fits_directory, test_set,
                                                  min_energy_plot = -20.0,
                                                  max_energy_plot = 20.0,
                                                  split_energy = 0.5)

### 3.10 Add files to MBX

In [ ]:
help(potential_fitting.generate_MBX_files)

In [ ]:
potential_fitting.generate_MBX_files(trim_settings, config, mon_ids, 
                                     do_ttmnrg=False, mbnrg_fits_path=mbnrg_fits_directory,  
                                     MBX_HOME = None, version = "v1")